# HydrAI — Tune One Tree Surrogate (Exit + Full PFR Evolution)

Combined workflow: tune one selected tree model on inlet→outlet exit-plane data and on full axial/PFR evolution data.

Prerequisites: Run Main_2 and Main_3 so `data/processed/features_targets_*.pkl` exists.

---

## Overfitting Control Strategy

Overfitting occurs when a model memorises training data but fails to generalise. This notebook employs:

1. Train/Test Split (80/20): Model never sees test data during training — final metrics reflect real-world performance.
2. Cross-Validation (CV=3–5): During hyperparameter tuning, each candidate is validated on multiple folds to detect overfitting early.
3. Regularisation Parameters:
   - `max_depth`: limits tree depth (prevents memorising noise)
   - `min_samples_split`, `min_samples_leaf`: requires minimum samples to split/form leaves
   - XGBoost: `reg_alpha` (L1), `reg_lambda` (L2), `gamma` (min loss reduction)
4. Early Stopping (XGBoost): optional, stops adding trees when validation score plateaus.
5. Ensemble Averaging: tree ensembles (RF, GB) inherently reduce variance vs single trees.

Key diagnostic: Compare Train R² vs Test R². A large gap (e.g., Train=0.999, Test=0.90) signals overfitting → increase regularisation or gather more data.

---

Sections:
1. Setup & imports
2. Paths & flags
3. Load data
4. Features & targets (exit-plane)
5. Train/test split & scaling
6. Train selected model
7. Hyperparameter tuning (exit-plane; optional via `IF_HYPERPARAM_TUNING`)
8. Full-profile axial/PFR evolution training (reuses exit-plane hyperparameters)
9. Cantera vs ML — full axial evolution preview figures
10. Test-set evaluation & model comparison
11. Actual vs Predicted scatter plots (state variables)
12. Species chemistry analysis (lumped groups)
13. Speed report (Cantera vs ML inference)
14. Export tuned models

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import os
import re
import sys
import glob
import json
import joblib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    median_absolute_error,
    max_error,
)
from sklearn.tree import DecisionTreeRegressor

try:
    from skopt import BayesSearchCV
    from skopt.space import Real, Integer, Categorical
except Exception:
    BayesSearchCV = None
    Real = Integer = Categorical = None

# Project root
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "src").exists() else current_dir.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.utils.plot_style import setup_matplotlib
from src.utils.run_log import start_run_log
from src.ml.dataframe_pickle import load_portable_pickle

setup_matplotlib()
start_run_log('Main_5_train_evaluate_tune_tree_model_evolution')
print("Libraries imported successfully.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. PATHS & FLAGS
# ═══════════════════════════════════════════════════════════════════════════════

# I/O flags
IF_PLOT_SHOWN        = True
IF_PLOT_EXPORT       = True
IF_MODEL_EXPORT      = True

# Workflows: exit-plane BayesSearchCV when IF_HYPERPARAM_TUNING; full-profile
# training when TRAIN_FULL_PROFILE. Which model to tune, and the tuning/row-cap
# budgets, are config-driven (Section 3) so no numbers need editing here.
IF_HYPERPARAM_TUNING = False
TRAIN_FULL_PROFILE   = True

# Paths
CONFIG_PATH        = project_root / "configs" / "ml" / "main5_tree_tuning_config.json"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
EXPORT_DIR         = project_root / "models" / "tree_tuned"
FIG_DIR            = project_root / "outputs" / "figures" / "Main_5_train_evaluate_tune_tree_model_evolution"

# Display names for models (used in plots throughout)
DISPLAY_NAMES = {
    'random_forest': 'Random Forest',
    'gradient_boosting': 'Gradient Boosting',
    'xgboost': 'XGBoost',
    'adaboost': 'AdaBoost',
}

print(f"Plot: {IF_PLOT_SHOWN}  |  Export figs: {IF_PLOT_EXPORT}  |  Export models: {IF_MODEL_EXPORT}")
print(f"Workflows: exit=True, full_profile={TRAIN_FULL_PROFILE}, tuning={IF_HYPERPARAM_TUNING}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3. LOAD CONFIG & DATA
# ═══════════════════════════════════════════════════════════════════════════════

# ── When this notebook actually uses main5_tree_tuning_config.json ─────────────────
# This cell is the *only* place the JSON file is consulted; nothing else in
# Main_5 re-reads it. It runs on Kernel > Run All, or whenever you re-execute
# this single cell. Keys consumed by Main_5:
#   - top level:           test_size, random_state, model_to_tune, full_profile_max_rows
#   - random_forest.*      n_estimators, max_depth, min_samples_leaf
#   - gradient_boosting.*  n_estimators, max_depth, min_samples_leaf
#   - xgboost.*            n_estimators, max_depth, learning_rate, subsample,
#                          colsample_bytree, reg_alpha, reg_lambda
#   - adaboost.*           n_estimators, learning_rate, max_depth
#   - tuning.*             n_iter, patience, min_delta, cv, scoring (BayesSearchCV budget)
# If the file is missing, or a key is omitted, the `config.get(..., default)`
# calls below fall back to the inline defaults shown as the second argument.
# The neural_network.* block is read by Main_6 / Main_7 only and is ignored here.
# Edit main5_tree_tuning_config.json and re-run this cell to change behaviour;
# no kernel restart needed.

if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
else:
    config = {}
    print(f"[WARN] Config not found: {CONFIG_PATH}. Using defaults.")

TEST_SIZE    = config.get("test_size", 0.2)
RANDOM_STATE = config.get("random_state", 42)
RF_CONFIG    = config.get("random_forest", {"n_estimators": 100, "max_depth": 20})
GB_CONFIG    = config.get("gradient_boosting", {"n_estimators": 150, "max_depth": 5})
# XGBoost library-style defaults (not tuned optimum); `xgboost` keys in json override.
_DEFAULT_XGB = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}
XGB_CONFIG = {**_DEFAULT_XGB, **config.get("xgboost", {})}
ADA_CONFIG   = config.get("adaboost", {"n_estimators": 200, "learning_rate": 0.1, "max_depth": 6})

MODEL_TO_TUNE = config.get("model_to_tune", "xgboost")  # 'random_forest' | 'gradient_boosting' | 'xgboost' | 'adaboost'
MODELS_TO_TRAIN = [MODEL_TO_TUNE]

FULL_PROFILE_MAX_ROWS = config.get("full_profile_max_rows", None)  # e.g. 50000 for a quick full-profile smoke test

TUNING_CONFIG    = config.get("tuning", {})
TUNING_N_ITER    = int(TUNING_CONFIG.get("n_iter", 60))
TUNING_PATIENCE  = int(TUNING_CONFIG.get("patience", 10))
TUNING_MIN_DELTA = float(TUNING_CONFIG.get("min_delta", 5e-4))
TUNING_CV        = int(TUNING_CONFIG.get("cv", 3))
TUNING_SCORING   = TUNING_CONFIG.get("scoring", 'neg_mean_absolute_error')
# Alternative TUNING_SCORING options (set in main5_tree_tuning_config.json):
#   'neg_mean_absolute_error'            - MAE (default, robust to outliers)
#   'neg_mean_squared_error'             - MSE (penalizes large errors more)
#   'neg_root_mean_squared_error'        - RMSE (interpretable units like MAE)
#   'neg_mean_absolute_percentage_error' - MAPE (scale-independent %)
#   'r2'                                 - R² (explained variance, higher is better)
#   'neg_median_absolute_error'          - MedAE (very robust to outliers)
#   'neg_mean_squared_log_error'         - MSLE (for targets with exponential growth)

if IF_HYPERPARAM_TUNING and BayesSearchCV is None:
    raise ImportError("BayesSearchCV requested but scikit-optimize (skopt) is not installed. Install with: pip install scikit-optimize")

print(f"Config: test_size={TEST_SIZE}, random_state={RANDOM_STATE}")
print(f"Model to tune: {MODEL_TO_TUNE}")
print(f"Tuning: {IF_HYPERPARAM_TUNING}  |  Method=BayesSearchCV  |  N_ITER(max)={TUNING_N_ITER}  |  CV={TUNING_CV}  |  patience={TUNING_PATIENCE}  |  min_delta={TUNING_MIN_DELTA}")

# Data (latest features_targets_*.pkl)
pkl_files = sorted(glob.glob(str(PROCESSED_DATA_DIR / "features_targets_*.pkl")), reverse=True)
if not pkl_files:
    raise FileNotFoundError(f"No features_targets_*.pkl in {PROCESSED_DATA_DIR}. Run Main_3 first.")

DATA_FILE = pkl_files[0]
loaded = load_portable_pickle(DATA_FILE)
df_features = loaded["df_features"]
df_target = loaded["df_target"]

print(f"Data: {Path(DATA_FILE).name}")
print(f"  Features: {df_features.shape[0]:,} rows × {df_features.shape[1]} cols")
print(f"  Targets:  {df_target.shape[0]:,} rows × {df_target.shape[1]} cols")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4. FEATURES & TARGETS
# ═══════════════════════════════════════════════════════════════════════════════
# Defines shared features/targets. Exit-plane data are extracted here;
# full-profile data are prepared later for axial/PFR evolution tuning.

feature_cols = [
    "initial_temperature_K", "initial_pressure_Pa",
    "reactor_length_m", "reactor_diameter_m",
    "mass_flow_rate_kgps", "heat_flux_Wm2",
    "z_position_m", "relative_position",
]
if "reactant_type" in df_features.columns:
    feature_cols.append("reactant_type")
feature_cols = [c for c in feature_cols if c in df_features.columns]

# Run-identifying columns (exclude spatial coords)
run_cols = [c for c in feature_cols if c not in ("z_position_m", "relative_position")]

# Primary targets (state + thermo)
primary_targets = [
    "temperature_K", "pressure_Pa", "velocity_ms", "density_kgm3",
    "mean_molecular_weight_kgkmol", "heat_capacity_cp_JkgK", "heat_capacity_cv_JkgK",
    "enthalpy_Jkg", "thermal_conductivity_WmK",
]
state_target_cols = [c for c in primary_targets if c in df_target.columns]

# Species targets: mass fractions only — individual Y_* or Y_lump_* from Main_3
_lump_cols = [c for c in df_target.columns if c.startswith('Y_lump_')]
if _lump_cols:
    species_cols = sorted(_lump_cols)
else:
    species_cols = [c for c in df_target.columns if c.startswith('Y_')]

# All targets for training
target_cols = state_target_cols + species_cols

# ═══════════════════════════════════════════════════════════════════════════════
# Chemistry grouping for plots: Y_lump_chem_* / Y_lump_carbon_* map 1:1; else classify Y_*
# ═══════════════════════════════════════════════════════════════════════════════
def classify_species_chemistry(species_name):
    """Classify species by chemical role."""
    name = species_name[2:] if species_name.startswith('Y_') else species_name
    name_base = name.split('(')[0]
    
    if name_base == 'H2':
        return 'hydrogen'
    if name_base in {'Water', 'Ar', 'He', 'Ne', 'N2', 'H'}:
        return 'diluent'
    if name_base == 'C6H14':
        return 'feedstock'
    if name_base in {'C2H4', 'C3H6', 'C4H6', 'C4H8'}:
        return 'olefins'
    if name_base in {'Benzene', 'Toluene', 'Styrene', 'C8H10'}:
        return 'aromatics'
    if name_base in {'CH4', 'CC', 'CCC'}:
        return 'paraffins'
    if name_base.startswith('C#C') or name_base == 'C2H2':
        return 'coke_precursors'
    if name_base in {'C5H6', 'C5H5', 'C3H4', 'C4H4', 'C4H5'}:
        return 'coke_precursors'
    # Highly unsaturated C6+
    match = re.match(r'^C(\d+)H(\d+)', name_base)
    if match:
        c, h = int(match.group(1)), int(match.group(2))
        if c >= 6 and h / c < 1.5:
            return 'coke_precursors'
    # Radicals
    if name_base in {'CH3', 'C2H3', 'C2H5', 'C3H5', 'C3H7', 'C4H7', 'C4H9', 
                     'C5H7', 'C5H9', 'C5H11', 'C6H7', 'C6H9', 'C6H11', 'C6H13',
                     'C7H9', 'C7H11', 'C3H3'}:
        return 'radicals'
    return 'other'

chemistry_groups = {}
if species_cols and all(c.startswith('Y_lump_chem_') for c in species_cols):
    for c in species_cols:
        role = c[len('Y_lump_chem_'):]
        chemistry_groups[role] = [c]
elif species_cols and all(c.startswith('Y_lump_carbon_') for c in species_cols):
    for c in species_cols:
        suf = c[len('Y_lump_carbon_'):]
        # Keep carbon-number labels for Cn buckets, but map inert explicitly to 'inert'.
        role = 'inert' if str(suf).lower() == 'inert' else f'carbon_{suf}'
        chemistry_groups[role] = [c]
else:
    for col in species_cols:
        role = classify_species_chemistry(col)
        chemistry_groups.setdefault(role, []).append(col)

print("Species / lump columns by chemistry role:")
for role, cols in sorted(chemistry_groups.items()):
    print(f"  {role}: {len(cols)} column(s)")

# Extract exit-plane rows (max relative_position per run)
exit_idx = df_features.groupby(run_cols, dropna=False)["relative_position"].idxmax().values
X_exit = df_features.loc[exit_idx, feature_cols].copy()
y_exit = df_target.loc[exit_idx, target_cols].copy()

# Encode categorical (reactant_type)
label_encoder = None
if "reactant_type" in X_exit.columns:
    label_encoder = LabelEncoder()
    X_exit["reactant_type"] = label_encoder.fit_transform(X_exit["reactant_type"].astype(str))

print(f"\nExit-plane dataset: {len(X_exit):,} samples")
print(f"  Input features ({len(feature_cols)}): {feature_cols}")
print(f"  State targets ({len(state_target_cols)}): {state_target_cols}")
print(f"  Species / lump targets: {len(species_cols)} columns")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5. TRAIN/TEST SPLIT & SCALING
# ═══════════════════════════════════════════════════════════════════════════════
# 80/20 split ensures models are evaluated on unseen data.
# StandardScaler normalises features (mean=0, std=1) — fitted on train only.

X_train, X_test, y_train, y_test = train_test_split(
    X_exit, y_exit, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {len(X_train):,} ({100*(1-TEST_SIZE):.0f}%)  |  Test: {len(X_test):,} ({100*TEST_SIZE:.0f}%)")
print(f"Targets: {len(target_cols)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 6. TRAIN MODELS (DEFAULT PARAMS)
# ═══════════════════════════════════════════════════════════════════════════════
ALL_MODEL_KEYS = ['random_forest', 'gradient_boosting', 'xgboost', 'adaboost']
keys_to_train = ALL_MODEL_KEYS if MODELS_TO_TRAIN is None else MODELS_TO_TRAIN

models = {}

if 'random_forest' in keys_to_train:
    print("Training Random Forest...")
    base_rf = RandomForestRegressor(
        n_estimators=RF_CONFIG.get("n_estimators", 100),
        max_depth=RF_CONFIG.get("max_depth", 20),
        min_samples_leaf=RF_CONFIG.get("min_samples_leaf", 1),
        random_state=RANDOM_STATE, n_jobs=1
    )
    models['random_forest'] = MultiOutputRegressor(base_rf, n_jobs=-1).fit(X_train_s, y_train)
    print("  Random Forest done.")

if 'gradient_boosting' in keys_to_train:
    print("Training Gradient Boosting...")
    base_gb = GradientBoostingRegressor(
        n_estimators=GB_CONFIG.get("n_estimators", 150),
        max_depth=GB_CONFIG.get("max_depth", 5),
        min_samples_leaf=GB_CONFIG.get("min_samples_leaf", 1),
        random_state=RANDOM_STATE
    )
    models['gradient_boosting'] = MultiOutputRegressor(base_gb, n_jobs=-1).fit(X_train_s, y_train)
    print("  Gradient Boosting done.")

if 'xgboost' in keys_to_train:
    print("Training XGBoost...")
    base_xgb = xgb.XGBRegressor(
        n_estimators=XGB_CONFIG.get("n_estimators", 100),
        max_depth=XGB_CONFIG.get("max_depth", 6),
        learning_rate=XGB_CONFIG.get("learning_rate", 0.3),
        subsample=XGB_CONFIG.get("subsample", 1.0),
        colsample_bytree=XGB_CONFIG.get("colsample_bytree", 1.0),
        reg_alpha=XGB_CONFIG.get("reg_alpha", 0.0),
        reg_lambda=XGB_CONFIG.get("reg_lambda", 1.0),
        random_state=RANDOM_STATE, n_jobs=1
    )
    models['xgboost'] = MultiOutputRegressor(base_xgb, n_jobs=-1).fit(X_train_s, y_train)
    print("  XGBoost done.")

if 'adaboost' in keys_to_train:
    print("Training AdaBoost...")
    base_ada = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(
            max_depth=ADA_CONFIG.get("max_depth", 6),
            random_state=RANDOM_STATE
        ),
        n_estimators=ADA_CONFIG.get("n_estimators", 200),
        learning_rate=ADA_CONFIG.get("learning_rate", 0.1),
        random_state=RANDOM_STATE
    )
    models['adaboost'] = MultiOutputRegressor(base_ada, n_jobs=-1).fit(X_train_s, y_train)
    print("  AdaBoost done.")

print(f"\nTrained models: {list(models.keys())}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 7. HYPERPARAMETER TUNING (OPTIONAL)
# ═══════════════════════════════════════════════════════════════════════════════
# BayesSearchCV with cross-validation finds good hyperparameters while
# avoiding overfitting (CV score penalises models that don't generalise).

if IF_HYPERPARAM_TUNING:
    if BayesSearchCV is None:
        raise ImportError("BayesSearchCV requires scikit-optimize. Install with: pip install scikit-optimize")
    method_name = 'BayesSearchCV'

    print("=" * 60)
    print(f"HYPERPARAMETER TUNING ({method_name})")
    print(f"  N_ITER={TUNING_N_ITER}, CV={TUNING_CV}, scoring={TUNING_SCORING}")
    print("=" * 60)

    bayes_spaces = {
            'random_forest': {
                'estimator__n_estimators': Integer(100, 300),
                'estimator__max_depth': Categorical([10, 15, 20, 25, None]),
                'estimator__min_samples_split': Integer(2, 10),
                'estimator__min_samples_leaf': Integer(1, 4),
            },
            'gradient_boosting': {
                'estimator__n_estimators': Integer(100, 300),
                'estimator__learning_rate': Real(0.01, 0.1, prior='log-uniform'),
                'estimator__max_depth': Integer(3, 6),
                'estimator__min_samples_leaf': Integer(1, 4),
                'estimator__subsample': Real(0.7, 1.0),
            },
            'xgboost': {
                'estimator__n_estimators': Integer(100, 400),
                'estimator__max_depth': Integer(3, 7),
                'estimator__learning_rate': Real(0.01, 0.1, prior='log-uniform'),
                'estimator__subsample': Real(0.7, 1.0),
                'estimator__colsample_bytree': Real(0.7, 1.0),
                'estimator__reg_alpha': Real(1e-6, 0.1, prior='log-uniform'),
                'estimator__reg_lambda': Real(1.0, 5.0),
            },
            'adaboost': {
                'estimator__n_estimators': Integer(100, 300),
                'estimator__learning_rate': Real(0.01, 0.2, prior='log-uniform'),
                'estimator__estimator__max_depth': Integer(3, 8),
            },
        }

    def build_base(key):
        if key == 'random_forest':
            return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
        if key == 'gradient_boosting':
            return GradientBoostingRegressor(random_state=RANDOM_STATE)
        if key == 'xgboost':
            return xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=1)
        if key == 'adaboost':
            return AdaBoostRegressor(
                estimator=DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
                random_state=RANDOM_STATE
            )

    def convergence_callback(result):
        """Stop BayesSearch when the best CV score has not improved by TUNING_MIN_DELTA
        over the last TUNING_PATIENCE consecutive iterations (minimal-return / convergence guard).
        BayesSearchCV minimises the objective, so lower func_vals = better score."""
        scores = np.array(result.func_vals)
        if len(scores) < TUNING_PATIENCE + 1:
            return False
        best_so_far = np.minimum.accumulate(scores)
        improvement = best_so_far[-(TUNING_PATIENCE + 1)] - best_so_far[-1]
        if improvement < TUNING_MIN_DELTA:
            print(
                f"    [Early stop] No improvement >{TUNING_MIN_DELTA:.4g} in last "
                f"{TUNING_PATIENCE} iters — stopped at iter {len(scores)}."
            )
            return True
        return False

    exit_best_params = {}

    for key in models.keys():
        if key not in bayes_spaces:
            continue
        print(f"\n  Tuning {key}...")
        pipe = MultiOutputRegressor(build_base(key), n_jobs=-1)

        search = BayesSearchCV(
            estimator=pipe,
            search_spaces=bayes_spaces[key],
            n_iter=TUNING_N_ITER,
            cv=TUNING_CV,
            scoring=TUNING_SCORING,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=0,
        )

        search.fit(X_train_s, y_train, callback=convergence_callback)
        score = -search.best_score_ if TUNING_SCORING.startswith('neg_') else search.best_score_
        print(f"    Best CV score: {score:.4f}")
        print(f"    Best params: {search.best_params_}")
        models[key] = search.best_estimator_
        if key == MODEL_TO_TUNE:
            exit_best_params = dict(search.best_params_)

    if not exit_best_params and MODEL_TO_TUNE in models:
        exit_best_params = {
            k: v for k, v in models[MODEL_TO_TUNE].get_params().items()
            if k.startswith('estimator__')
        }

    print("\nTuning complete. Models updated with best estimators.")
else:
    print("Hyperparameter tuning disabled.")
    exit_best_params = {}
    if MODEL_TO_TUNE in models:
        exit_best_params = {
            k: v for k, v in models[MODEL_TO_TUNE].get_params().items()
            if k.startswith('estimator__')
        }


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 8. FULL AXIAL / PFR EVOLUTION TRAINING
# ═══════════════════════════════════════════════════════════════════════════════
# This trains the same selected model on every axial row, with relative_position
# included as an input so the surrogate can predict reactor evolution along z/L.
# Split is by simulation run, not by row, to avoid train/test leakage along the same PFR profile.

if TRAIN_FULL_PROFILE:
    print("=" * 70)
    print("FULL PFR EVOLUTION TRAINING  (params transferred from exit-plane tuning)")
    print(f"  Model={MODEL_TO_TUNE}  |  no separate tuning run for axial evolution")
    print("=" * 70)

    def build_full_base(key):
        if key == 'random_forest':
            return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
        if key == 'gradient_boosting':
            return GradientBoostingRegressor(random_state=RANDOM_STATE)
        if key == 'xgboost':
            return xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=1)
        if key == 'adaboost':
            return AdaBoostRegressor(
                estimator=DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
                random_state=RANDOM_STATE,
            )
        raise ValueError(f"Unsupported MODEL_TO_TUNE: {key}")

    X_full = df_features.loc[:, feature_cols].copy()
    y_full = df_target.loc[:, target_cols].copy()

    full_label_encoder = None
    if "reactant_type" in X_full.columns:
        full_label_encoder = LabelEncoder()
        X_full["reactant_type"] = full_label_encoder.fit_transform(X_full["reactant_type"].astype(str))

    # Use run-level split so the model is tested on unseen axial profiles.
    full_run_id = df_features.groupby(run_cols, dropna=False).ngroup()
    unique_runs = np.array(sorted(pd.unique(full_run_id)))
    full_train_runs, full_test_runs = train_test_split(
        unique_runs,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    full_train_mask = full_run_id.isin(full_train_runs)
    full_test_mask = full_run_id.isin(full_test_runs)

    X_full_train = X_full.loc[full_train_mask].copy()
    X_full_test = X_full.loc[full_test_mask].copy()
    y_full_train = y_full.loc[full_train_mask].copy()
    y_full_test = y_full.loc[full_test_mask].copy()

    if FULL_PROFILE_MAX_ROWS is not None and len(X_full_train) + len(X_full_test) > FULL_PROFILE_MAX_ROWS:
        n_train_keep = int(FULL_PROFILE_MAX_ROWS * (1 - TEST_SIZE))
        n_test_keep = FULL_PROFILE_MAX_ROWS - n_train_keep
        X_full_train = X_full_train.sample(n=min(n_train_keep, len(X_full_train)), random_state=RANDOM_STATE)
        y_full_train = y_full_train.loc[X_full_train.index]
        X_full_test = X_full_test.sample(n=min(n_test_keep, len(X_full_test)), random_state=RANDOM_STATE)
        y_full_test = y_full_test.loc[X_full_test.index]
        print(f"  Subsampled full-profile rows to {len(X_full_train) + len(X_full_test):,}")

    full_scaler = StandardScaler()
    X_full_train_s = full_scaler.fit_transform(X_full_train)
    X_full_test_s = full_scaler.transform(X_full_test)

    # Reuse exit-plane hyperparameters (from BayesSearchCV or from cell 6 defaults when tuning is off).
    full_profile_model = MultiOutputRegressor(build_full_base(MODEL_TO_TUNE), n_jobs=-1)
    transfer = globals().get('exit_best_params') or {}
    if not transfer and MODEL_TO_TUNE in models:
        transfer = {
            k: v for k, v in models[MODEL_TO_TUNE].get_params().items()
            if k.startswith('estimator__')
        }
    if transfer:
        full_profile_model.set_params(**transfer)
        full_best_params = transfer
        print(f"  Transferred exit-plane params: {full_best_params}")
    else:
        full_best_params = 'base estimator defaults (run exit training/tuning cells first)'
        print("  Warning: no estimator__ params to transfer — using base defaults.")
    full_profile_model.fit(X_full_train_s, y_full_train)
    full_best_cv_score = None

    full_profile_models = {MODEL_TO_TUNE: full_profile_model}

    y_full_train_pred = full_profile_model.predict(X_full_train_s)
    y_full_test_pred = full_profile_model.predict(X_full_test_s)

    from src.utils.profile_predictions import anchor_inlet_profile_predictions

    y_full_test_pred, _n_inlet_anchored = anchor_inlet_profile_predictions(
        y_full_test_pred,
        y_full_test.values,
        full_run_id.loc[X_full_test.index].to_numpy(),
        X_full_test["relative_position"].to_numpy(),
    )
    print(
        f"  Inlet BC anchoring (test): {_n_inlet_anchored} row(s) "
        f"at min x/L per run — axial plots share Cantera inlet state."
    )

    full_profile_metrics = {
        'Train_R2': r2_score(y_full_train, y_full_train_pred, multioutput='uniform_average'),
        'Test_R2': r2_score(y_full_test, y_full_test_pred, multioutput='uniform_average'),
        'Test_MAE': mean_absolute_error(y_full_test, y_full_test_pred, multioutput='uniform_average'),
        'Test_RMSE': np.sqrt(mean_squared_error(y_full_test, y_full_test_pred, multioutput='uniform_average')),
        'Best_CV_Score': full_best_cv_score,
        'Best_Params': full_best_params,
        'Train_Rows': len(X_full_train),
        'Test_Rows': len(X_full_test),
        'Train_Runs': len(full_train_runs),
        'Test_Runs': len(full_test_runs),
    }

    print("\nFull-profile training complete")
    print(f"  Train rows/runs: {len(X_full_train):,} / {len(full_train_runs):,}")
    print(f"  Test rows/runs : {len(X_full_test):,} / {len(full_test_runs):,}")
    print(f"  Test R²     : {full_profile_metrics['Test_R2']:.4f}")
    print(f"  Test MAE    : {full_profile_metrics['Test_MAE']:.4g}")
    print(f"  Params used : {full_profile_metrics['Best_Params']}")

    # State / thermo error across the full axial evolution test runs.
    full_state_nmae = {}
    y_full_test_np = y_full_test.values if hasattr(y_full_test, 'values') else y_full_test
    for tgt in state_target_cols:
        tgt_idx = target_cols.index(tgt)
        yt = y_full_test_np[:, tgt_idx]
        yp = y_full_test_pred[:, tgt_idx]

        mae = mean_absolute_error(yt, yp)
        yrange = np.max(yt) - np.min(yt)
        if yrange > 1e-12:
            nmae = mae / yrange
        else:
            ymean = np.mean(np.abs(yt))
            nmae = mae / ymean if ymean > 1e-12 else 0.0
        full_state_nmae[tgt] = nmae * 100.0

    full_profile_metrics['State_NMAE_%'] = full_state_nmae

    _state_labels = {
        'temperature_K': 'T',
        'pressure_Pa': 'Pressure (bar)',
        'velocity_ms': 'Velocity',
        'density_kgm3': 'Density',
        'mean_molecular_weight_kgkmol': 'MW',
        'heat_capacity_cp_JkgK': 'Cp',
        'heat_capacity_cv_JkgK': 'Cv',
        'enthalpy_Jkg': 'Enthalpy',
        'thermal_conductivity_WmK': 'Therm. cond.',
    }

    fig, ax = plt.subplots(figsize=(10, 4.5))
    labels = [_state_labels.get(t, t) for t in full_state_nmae.keys()]
    ax.bar(labels, full_state_nmae.values(), color='slateblue', edgecolor='white', linewidth=0.5, alpha=0.85)
    ax.axhline(5, color='g', linestyle='--', alpha=1, label='NMAE <= 5%: excellent')
    ax.axhline(10, color='b', linestyle='--', alpha=1, label='NMAE <= 10%: good')
    ax.axhline(20, color='r', linestyle='--', alpha=1, label='NMAE > 20%: weak')
    ax.set_ylabel('Normalized MAE (%)')
    ax.set_xlabel('State / thermo target')
    ax.set_title(f'Full-profile error by state / thermo target (Test Runs)\n{DISPLAY_NAMES.get(MODEL_TO_TUNE, MODEL_TO_TUNE)}')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(loc='upper left', fontsize=8, ncol=3)
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'full_profile_state_thermo_target_nmae.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

    print("\nFULL-PROFILE STATE / THERMO ERROR (Normalized MAE %, Test Runs)")
    df_full_state_nmae = pd.DataFrame({
        'Target': list(full_state_nmae.keys()),
        'NMAE_%': list(full_state_nmae.values()),
    })
    print(df_full_state_nmae.to_string(index=False))
else:
    print("Full-profile workflow disabled (TRAIN_FULL_PROFILE=False).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9. CANTERA VS ML — FULL AXIAL EVOLUTION ALONG THE REACTOR
# ═══════════════════════════════════════════════════════════════════════════════
# For a few unseen test runs, overlay the Cantera/test profile and the ML-predicted
# profile versus x/L. Predictions are inlet-anchored in §8 (min x/L per run = Cantera state).
# Vertical markers show 25%, 50%, 75%, and 100% reactor length.

AXIAL_COMPARISON_POSITIONS = [0.0, 0.25, 0.50, 0.75, 1.00]
AXIAL_PROFILE_N_RUNS = 6
AXIAL_FEED_LABEL_OVERRIDE = 'nHexane'
AXIAL_PROFILE_TARGETS = [
    c for c in [
        'temperature_K',
        'pressure_Pa',
        'velocity_ms',
        'density_kgm3',
        'residence_time_s',
        'mean_molecular_weight_kgkmol',
    ]
    if c in target_cols
]

if (
    'full_profile_model' in globals()
    and 'X_full_test' in globals()
    and 'y_full_test' in globals()
    and 'y_full_test_pred' in globals()
    and 'full_run_id' in globals()
    and 'relative_position' in X_full_test.columns
    and AXIAL_PROFILE_TARGETS
):
    full_plot_meta = X_full_test[['relative_position']].copy()
    full_plot_meta['_run_id'] = full_run_id.loc[X_full_test.index].to_numpy()
    if 'reactant_type' in X_full_test.columns:
        if 'full_label_encoder' in globals() and full_label_encoder is not None:
            try:
                _codes = X_full_test['reactant_type'].astype(int).to_numpy()
                _labels = full_label_encoder.inverse_transform(_codes)
                full_plot_meta['_feed'] = _labels
            except Exception:
                full_plot_meta['_feed'] = X_full_test['reactant_type'].astype(str).to_numpy()
        else:
            full_plot_meta['_feed'] = X_full_test['reactant_type'].astype(str).to_numpy()
    else:
        full_plot_meta['_feed'] = 'unknown_feed'

    full_actual_df = y_full_test.loc[:, target_cols].copy()
    full_pred_df = pd.DataFrame(y_full_test_pred, index=X_full_test.index, columns=target_cols)

    selected_profile_runs = full_plot_meta['_run_id'].drop_duplicates().head(AXIAL_PROFILE_N_RUNS).to_list()
    n_rows = len(AXIAL_PROFILE_TARGETS)
    n_cols = len(selected_profile_runs)

    if n_cols > 0:
        fig, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(5.2 * n_cols, 3.2 * n_rows),
            squeeze=False,
        )

        station_rows = []
        target_labels = {
            'temperature_K': 'Temperature (K)',
            'pressure_Pa': 'Pressure (bar)',
            'velocity_ms': 'Velocity (m/s)',
            'density_kgm3': 'Density (kg/m3)',
            'residence_time_s': 'Residence time (s)',
            'mean_molecular_weight_kgkmol': 'Mean molecular weight (kg/kmol)',
        }

        for col_idx, run_id in enumerate(selected_profile_runs):
            run_mask = full_plot_meta['_run_id'] == run_id
            run_index = full_plot_meta.index[run_mask]
            _feed_vals = full_plot_meta.loc[run_index, '_feed']
            feed_label = str(_feed_vals.iloc[0]) if len(_feed_vals) else 'unknown_feed'
            if AXIAL_FEED_LABEL_OVERRIDE:
                feed_label = AXIAL_FEED_LABEL_OVERRIDE
            x_rel = full_plot_meta.loc[run_index, 'relative_position'].astype(float)
            order = np.argsort(x_rel.to_numpy())
            x_sorted = x_rel.to_numpy()[order]
            sorted_index = run_index[order]

            for row_idx, tgt in enumerate(AXIAL_PROFILE_TARGETS):
                ax = axes[row_idx, col_idx]
                actual = full_actual_df.loc[sorted_index, tgt].to_numpy(dtype=float)
                pred = full_pred_df.loc[sorted_index, tgt].to_numpy(dtype=float)

                if tgt == 'pressure_Pa':
                    actual = actual / 1e5
                    pred = pred / 1e5

                ax.plot(x_sorted, actual, 'b-', lw=2, label='Cantera / test')
                model_name = DISPLAY_NAMES.get(MODEL_TO_TUNE, MODEL_TO_TUNE)
                ax.plot(
                    x_sorted,
                    pred,
                    'r-',
                    lw=2,
                    label=f"ML prediction ({model_name})",
                )

                for station in AXIAL_COMPARISON_POSITIONS:
                    ax.axvline(station, color='k', linestyle='--', lw=1.25, alpha=0.5)

                ax.set_xlim(0, 1.02)
                ax.set_title(f"Run {run_id} | Feed={feed_label} — {target_labels.get(tgt, tgt)}", fontsize=9)
                if row_idx == n_rows - 1:
                    ax.set_xlabel('x/L')
                if col_idx == 0:
                    ax.set_ylabel(target_labels.get(tgt, tgt))
                if row_idx == 0 and col_idx == 0:
                    ax.legend(loc='best', fontsize=8)

                for station in AXIAL_COMPARISON_POSITIONS:
                    nearest_idx = (x_rel - station).abs().idxmin()
                    actual_station = float(full_actual_df.loc[nearest_idx, tgt])
                    pred_station = float(full_pred_df.loc[nearest_idx, tgt])
                    if tgt == 'pressure_Pa':
                        actual_station /= 1e5
                        pred_station /= 1e5
                    station_rows.append({
                        'Run_ID': run_id,
                        'x_over_L_requested': station,
                        'x_over_L_used': float(full_plot_meta.loc[nearest_idx, 'relative_position']),
                        'Target': target_labels.get(tgt, tgt),
                        'Actual': actual_station,
                        'Predicted': pred_station,
                        'Abs_Error': abs(pred_station - actual_station),
                    })

        plt.suptitle(
            f'Full-profile axial evolution: Cantera vs ML ({DISPLAY_NAMES.get(MODEL_TO_TUNE, MODEL_TO_TUNE)})',
            y=1.01,
        )
        plt.tight_layout()

        if IF_PLOT_EXPORT:
            FIG_DIR.mkdir(parents=True, exist_ok=True)
            fig.savefig(FIG_DIR / 'full_profile_cantera_vs_ml_axial_evolution.png', dpi=150, bbox_inches='tight')
        if IF_PLOT_SHOWN:
            plt.show()
        plt.close(fig)

        df_axial_station_comparison = pd.DataFrame(station_rows)
        print('CANTERA VS ML FULL-PROFILE COMPARISON AT SELECTED x/L STATIONS')
        print(df_axial_station_comparison.to_string(index=False, float_format=lambda x: f'{x:.5g}'))
    else:
        print('No full-profile test runs available for axial evolution plotting.')
else:
    # This branch runs if any prerequisite for the axial plot is missing — not only TRAIN_FULL_PROFILE.
    if not TRAIN_FULL_PROFILE:
        print(
            'Skipping Cantera vs ML axial plot: TRAIN_FULL_PROFILE=False '
            '(set True and run the full-profile training cell before this one).'
        )
    elif (
        'full_profile_model' not in globals()
        or 'y_full_test_pred' not in globals()
        or 'y_full_test' not in globals()
        or 'full_run_id' not in globals()
    ):
        print(
            'Skipping axial plot: full-profile variables missing — run the full-profile training cell '
            '(after upstream setup) without restarting the kernel before this cell.'
        )
    elif 'X_full_test' not in globals():
        print('Skipping axial plot: X_full_test undefined — re-run full-profile training.')
    elif 'relative_position' not in X_full_test.columns:
        print(
            'Skipping axial plot: full-profile features lack relative_position '
            '(it must stay in feature_cols for axial evolution).'
        )
    elif not AXIAL_PROFILE_TARGETS:
        print(
            'Skipping axial plot: AXIAL_PROFILE_TARGETS empty — none of the requested state columns '
            'appear in target_cols.'
        )
    else:
        print('Skipping axial plot — re-run full-profile training; if this persists, check upstream errors in that cell.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10. TEST-SET EVALUATION & MODEL COMPARISON
# ═══════════════════════════════════════════════════════════════════════════════
# Key metrics:
#   R² — explained variance (1.0 = perfect, <0 = worse than mean)
#   MAE — mean absolute error (interpretable units)
#   RMSE — root mean squared error (penalises large errors)
#   MAPE% — mean absolute percentage error (scale-independent)
#   MBE — mean bias error (positive = overprediction)
#
# Overfitting check: Train R² >> Test R² indicates overfitting.

def compute_metrics(y_true, y_pred, target_names):
    """Compute per-target metrics."""
    rows = []
    for i, tgt in enumerate(target_names):
        yt = y_true[:, i] if y_true.ndim > 1 else y_true
        yp = y_pred[:, i] if y_pred.ndim > 1 else y_pred
        mask = np.abs(yt) > 1e-12
        mape = np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100 if mask.any() else np.nan
        rows.append({
            'target': tgt,
            'R2': r2_score(yt, yp),
            'MAE': mean_absolute_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAPE_%': mape,
            'MBE': np.mean(yp - yt),
        })
    return pd.DataFrame(rows)

# Compute metrics for each model
results = []
for key, model in models.items():
    # Train metrics (for overfitting check)
    y_train_pred = model.predict(X_train_s)
    train_r2 = r2_score(y_train, y_train_pred, multioutput='uniform_average')
    
    # Test metrics
    y_test_pred = model.predict(X_test_s)
    test_r2 = r2_score(y_test, y_test_pred, multioutput='uniform_average')
    test_mae = mean_absolute_error(y_test, y_test_pred, multioutput='uniform_average')
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred, multioutput='uniform_average'))
    
    # Per-target MAPE
    y_test_np = y_test.values if hasattr(y_test, 'values') else y_test
    mask = np.abs(y_test_np) > 1e-12
    mape_per_target = np.abs((y_test_np - y_test_pred) / np.where(mask, y_test_np, 1)) * 100
    mape_per_target = np.where(mask, mape_per_target, np.nan)
    test_mape = np.nanmean(mape_per_target)
    
    results.append({
        'Model': DISPLAY_NAMES.get(key, key),
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Test_MAE': test_mae,
        'Test_RMSE': test_rmse,
        'Test_MAPE_%': test_mape,
        'Overfit_Gap': train_r2 - test_r2,
    })

df_results = pd.DataFrame(results).sort_values('Test_R2', ascending=False)

def _extract_best_model_info(best_model_obj):
    """Return compact info for the best model (counts + key hyperparameters)."""
    info = {
        'base_estimator': type(best_model_obj).__name__,
        'n_outputs': None,
        'n_total_estimators': None,
        'key_params': {},
    }

    # Most models here are wrapped in MultiOutputRegressor
    if hasattr(best_model_obj, 'estimators_') and len(best_model_obj.estimators_) > 0:
        first_est = best_model_obj.estimators_[0]
        info['base_estimator'] = type(first_est).__name__
        info['n_outputs'] = len(best_model_obj.estimators_)

        if hasattr(first_est, 'n_estimators'):
            n_per_output = getattr(first_est, 'n_estimators', None)
            if isinstance(n_per_output, (int, np.integer)):
                info['n_total_estimators'] = int(n_per_output) * info['n_outputs']

        params = first_est.get_params() if hasattr(first_est, 'get_params') else {}
        key_fields = [
            'n_estimators', 'max_depth', 'learning_rate', 'min_samples_split',
            'min_samples_leaf', 'subsample', 'colsample_bytree', 'reg_alpha', 'reg_lambda'
        ]
        info['key_params'] = {k: params[k] for k in key_fields if k in params}

    return info

print("=" * 80)
print("MODEL COMPARISON — Test Set Evaluation")
print("=" * 80)
print(df_results.to_string(index=False))
print()
print("Key: Overfit_Gap = Train_R² - Test_R² (large gap > 0.05 suggests overfitting)")
print("=" * 80)

# Best model summary with additional context
best_display = str(df_results.iloc[0]['Model'])
best_key = next((k for k, v in DISPLAY_NAMES.items() if v == best_display), None)
if best_key is not None and best_key in models:
    best_info = _extract_best_model_info(models[best_key])

    n_train = X_train.shape[0] if hasattr(X_train, 'shape') else len(X_train)
    n_test = X_test.shape[0] if hasattr(X_test, 'shape') else len(X_test)
    n_features = X_train.shape[1] if hasattr(X_train, 'shape') and len(X_train.shape) > 1 else None
    n_targets = y_train.shape[1] if hasattr(y_train, 'shape') and len(y_train.shape) > 1 else 1

    print(f"Best model: {best_display}")
    print(f"  Data: n_train={n_train:,}, n_test={n_test:,}, n_features={n_features}, n_targets={n_targets}")
    print(f"  Base estimator: {best_info['base_estimator']}")
    if best_info['n_outputs'] is not None:
        print(f"  Multi-output estimators: {best_info['n_outputs']}")
    if best_info['n_total_estimators'] is not None:
        print(f"  Approx total trees: {best_info['n_total_estimators']:,}")
    if best_info['key_params']:
        print(f"  Key hyperparameters: {best_info['key_params']}")
    print("=" * 80)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10b. PER-TARGET METRICS TABLE
# ═══════════════════════════════════════════════════════════════════════════════
# Detailed breakdown per target variable.

per_target_rows = []
for key, model in models.items():
    y_pred = model.predict(X_test_s)
    y_test_np = y_test.values if hasattr(y_test, 'values') else y_test
    for i, tgt in enumerate(target_cols):
        yt = y_test_np[:, i]
        yp = y_pred[:, i]
        mask = np.abs(yt) > 1e-12
        mape = np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100 if mask.any() else np.nan
        per_target_rows.append({
            'Model': DISPLAY_NAMES.get(key, key),
            'Target': tgt,
            'R2': r2_score(yt, yp),
            'MAE': mean_absolute_error(yt, yp),
            'RMSE': np.sqrt(mean_squared_error(yt, yp)),
            'MAPE_%': mape,
        })

df_per_target = pd.DataFrame(per_target_rows)

# Pivot for comparison
print("\nPer-target R² (Test Set):")
print("-" * 60)
pivot_r2 = df_per_target.pivot(index='Target', columns='Model', values='R2')
print(pivot_r2.round(4).to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 11. ACTUAL VS PREDICTED — SCATTER PLOTS (STATE VARIABLES ONLY)
# ═══════════════════════════════════════════════════════════════════════════════
# Perfect predictions lie on y=x diagonal. Spread around diagonal = error.
# Only plotting state targets here; species are analyzed in chemistry section.

# Only state variables for scatter plots (species handled in chemistry section)
scatter_targets = state_target_cols
n_scatter = len(scatter_targets)
n_models = len(models)

if n_scatter > 0 and n_models > 0:
    n_panels = n_models * n_scatter
    n_cols = 3
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.4 * n_cols, 3.6 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

    panel_idx = 0
    for key, model in models.items():
        y_pred = model.predict(X_test_s)

        for tgt in scatter_targets:
            ax = flat_axes[panel_idx]
            panel_idx += 1

            tgt_idx = target_cols.index(tgt)
            yt = y_test_np[:, tgt_idx]
            yp = y_pred[:, tgt_idx]

            tgt_label = tgt
            if tgt == 'pressure_Pa':
                yt = yt / 1e5
                yp = yp / 1e5
                tgt_label = 'Pressure (bar)'

            ax.scatter(yt, yp, alpha=0.25, s=10, edgecolors='none', c='b')
            lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
            ax.plot(lims, lims, 'r-', lw=2, label='y=x')
            ax.set_xlim(lims)
            ax.set_ylim(lims)

            r2 = r2_score(yt, yp)
            ax.set_title(f"{DISPLAY_NAMES.get(key, key)}\n{tgt_label}", fontsize=9)
            ax.text(0.05, 0.95, f"R²={r2:.4f}", transform=ax.transAxes, fontsize=8, va='top')
            ax.set_xlabel('Actual')
            ax.set_ylabel('Predicted')

    for ax in flat_axes[panel_idx:]:
        ax.set_visible(False)
    
    plt.suptitle('Actual vs Predicted — State Variables (Test Set)', y=1.01)
    plt.tight_layout()
    
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'exit_actual_vs_predicted_state_scatter.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)
else:
    print("No state targets or models to plot.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 12. SPECIES CHEMISTRY ANALYSIS (LUMPED GROUPS)
# ═══════════════════════════════════════════════════════════════════════════════
# Analyze model performance on chemistry-grouped species.
# Groups: olefins (primary products), aromatics (BTX), paraffins, coke_precursors, radicals, etc.

# Use the best model for species analysis
best_model_key = df_results.iloc[0]['Model'].lower().replace(' ', '_')
if best_model_key not in models:
    best_model_key = list(models.keys())[0]
best_model = models[best_model_key]
print(f"Using best model for species analysis: {DISPLAY_NAMES.get(best_model_key, best_model_key)}")

# Get predictions
y_test_pred = best_model.predict(X_test_s)
y_test_np = y_test.values if hasattr(y_test, 'values') else y_test

# ── Plot 1: Lumped yields comparison (actual vs predicted) ──────────────────────
_default_chem_order = ['olefins', 'aromatics', 'paraffins', 'coke_precursors', 'radicals', 'feedstock', 'hydrogen', 'diluent', 'other']
if chemistry_groups and all(k in _default_chem_order for k in chemistry_groups.keys()):
    chemistry_order = [c for c in _default_chem_order if c in chemistry_groups]
else:
    chemistry_order = sorted(chemistry_groups.keys())

# Calculate lumped yields for each chemistry group
lumped_actual = {}
lumped_pred = {}
for role in chemistry_order:
    cols = chemistry_groups[role]
    col_idx = [target_cols.index(c) for c in cols if c in target_cols]
    if col_idx:
        lumped_actual[role] = y_test_np[:, col_idx].sum(axis=1).mean()
        lumped_pred[role] = y_test_pred[:, col_idx].sum(axis=1).mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(lumped_actual))
width = 0.35
ax.bar(x - width/2, list(lumped_actual.values()), width, label='Actual', color='b', alpha=0.8)
ax.bar(x + width/2, list(lumped_pred.values()), width, label='Predicted (best model)', color='red', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(list(lumped_actual.keys()), rotation=0, ha='center')
ax.set_ylabel('Mean lumped mass fraction (Y)')
ax.set_title(f'Chemistry-grouped yields: Actual vs Predicted (Test Set)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}')
ax.legend()
plt.tight_layout()
if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'exit_chemistry_lumped_yields_comparison.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

# ── Plot 2: Normalized MAE per chemistry group ─────────────────────────────────
nmae_by_group = {}
for role in chemistry_order:
    cols = chemistry_groups[role]
    col_idx = [target_cols.index(c) for c in cols if c in target_cols]
    if col_idx:
        yt = y_test_np[:, col_idx].sum(axis=1)
        yp = y_test_pred[:, col_idx].sum(axis=1)

        mae = mean_absolute_error(yt, yp)
        yrange = np.max(yt) - np.min(yt)
        if yrange > 1e-12:
            nmae = mae / yrange
        else:
            # Fallback for near-constant targets
            ymean = np.mean(np.abs(yt))
            nmae = mae / ymean if ymean > 1e-12 else 0.0
        nmae_by_group[role] = nmae * 100.0  # percentage

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(nmae_by_group.keys(), nmae_by_group.values(), color='gray', edgecolor='white', linewidth=0.5)
ax.axhline(5, color='g', linestyle='--', alpha=1, label='NMAE <= 5%: excellent')
ax.axhline(10, color='b', linestyle='--', alpha=1, label='NMAE <= 10%: good')
ax.axhline(20, color='r', linestyle='--', alpha=1, label='NMAE > 20%: weak')
ax.set_ylabel('Normalized MAE (%)')
ax.set_xlabel('Chemistry group')
ax.set_title(f'Model error by chemistry group (Test Set)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'exit_chemistry_group_nmae.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

# ── Plot 3: Normalized MAE for state / thermo / aero targets ───────────────────
state_nmae = {}
for tgt in state_target_cols:
    tgt_idx = target_cols.index(tgt)
    yt = y_test_np[:, tgt_idx]
    yp = y_test_pred[:, tgt_idx]

    mae = mean_absolute_error(yt, yp)
    yrange = np.max(yt) - np.min(yt)
    if yrange > 1e-12:
        nmae = mae / yrange
    else:
        ymean = np.mean(np.abs(yt))
        nmae = mae / ymean if ymean > 1e-12 else 0.0
    state_nmae[tgt] = nmae * 100.0

_state_labels = {
    'temperature_K': 'Exit T',
    'pressure_Pa': 'Exit P (bar)',
    'velocity_ms': 'Velocity',
    'density_kgm3': 'Density',
    'mean_molecular_weight_kgkmol': 'MW',
    'heat_capacity_cp_JkgK': 'Cp',
    'heat_capacity_cv_JkgK': 'Cv',
    'enthalpy_Jkg': 'Enthalpy',
    'thermal_conductivity_WmK': 'Therm. cond.',
}

fig, ax = plt.subplots(figsize=(10, 4.5))
labels = [_state_labels.get(t, t) for t in state_nmae.keys()]
ax.bar(labels, state_nmae.values(), color='slateblue', edgecolor='white', linewidth=0.5, alpha=0.85)
ax.axhline(5, color='g', linestyle='--', alpha=1, label='NMAE <= 5%: excellent')
ax.axhline(10, color='b', linestyle='--', alpha=1, label='NMAE <= 10%: good')
ax.axhline(20, color='r', linestyle='--', alpha=1, label='NMAE > 20%: weak')
ax.set_ylabel('Normalized MAE (%)')
ax.set_xlabel('Exit state / thermo target')
ax.set_title(f'Error by exit state / thermo target (Test Set)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}')
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'exit_state_thermo_target_nmae.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

print("\nSTATE / THERMO TARGET ERROR (Normalized MAE %, Test Set)")
df_state_nmae = pd.DataFrame({
    'Target': list(state_nmae.keys()),
    'NMAE_%': list(state_nmae.values()),
})
print(df_state_nmae.to_string(index=False))

# ── Plot 4: Scatter plot — actual vs predicted for chemistry groups ────────────
# Single non-duplicative view: all available chemistry/carbon groups (dynamic grid).
all_groups = [g for g in chemistry_order if g in chemistry_groups]
if all_groups:
    n_panels = len(all_groups)
    n_cols = min(4, n_panels)
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.6 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    for i, role in enumerate(all_groups):
        ax = flat_axes[i]
        cols = chemistry_groups[role]
        col_idx = [target_cols.index(c) for c in cols if c in target_cols]
        if col_idx:
            yt = y_test_np[:, col_idx].sum(axis=1)
            yp = y_test_pred[:, col_idx].sum(axis=1)
            ax.scatter(yt, yp, alpha=0.25, s=10, edgecolors='none', c='b')
            lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
            ax.plot(lims, lims, 'r-', lw=2)
            ax.set_xlim(lims)
            ax.set_ylim(lims)
            r2 = r2_score(yt, yp)
            ax.set_title(f'{role.replace("_", " ").title()}\nR²={r2:.3f}', fontsize=9)
            ax.set_xlabel('Actual', fontsize=8)
            ax.set_ylabel('Predicted', fontsize=8)

    for j in range(n_panels, len(flat_axes)):
        flat_axes[j].set_visible(False)

    plt.suptitle(f'Lumped yields: Actual vs Predicted (All groups)\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}', y=1.02)
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'exit_chemistry_scatter_all_groups.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

# ── Plot 5: Best-model state targets (exit predictions) ───────────────────────
preferred_state_targets = ['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3']
state_scatter_targets = [t for t in preferred_state_targets if t in state_target_cols]
if not state_scatter_targets:
    state_scatter_targets = state_target_cols[:4]

if state_scatter_targets:
    n_panels = len(state_scatter_targets)
    n_cols = min(4, n_panels)
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.2 * n_cols, 4.2 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    for i, tgt in enumerate(state_scatter_targets):
        ax = flat_axes[i]
        tgt_idx = target_cols.index(tgt)
        yt = y_test_np[:, tgt_idx]
        yp = y_test_pred[:, tgt_idx]

        if tgt == 'pressure_Pa':
            yt = yt / 1e5
            yp = yp / 1e5
            unit_label = 'Pressure (bar)'
        elif tgt == 'temperature_K':
            unit_label = 'Temperature (K)'
        elif tgt == 'velocity_ms':
            unit_label = 'Velocity (m/s)'
        elif tgt == 'density_kgm3':
            unit_label = 'Density (kg/m³)'
        else:
            unit_label = tgt

        ax.scatter(yt, yp, alpha=0.25, s=10, edgecolors='none', c='b')
        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lims, lims, 'r-', lw=2)
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        r2 = r2_score(yt, yp)
        ax.set_title(f'{unit_label}\nR²={r2:.4f}', fontsize=10)
        ax.set_xlabel('Actual')
        ax.set_ylabel('Predicted')

    for j in range(n_panels, len(flat_axes)):
        flat_axes[j].set_visible(False)

    plt.suptitle(f'Best-model exit state predictions\n{DISPLAY_NAMES.get(best_model_key, best_model_key)}', y=1.02)
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'exit_state_scatter_best_model.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

# ── Summary table ───────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("CHEMISTRY GROUP PERFORMANCE (lumped mass fractions)")
print("=" * 60)
summary_rows = []
for role in chemistry_order:
    if role in lumped_actual and role in nmae_by_group:
        summary_rows.append({
            'Group': role,
            'N_species': len(chemistry_groups.get(role, [])),
            'Actual_Y': lumped_actual[role],
            'Predicted_Y': lumped_pred[role],
            'Error_%': (lumped_pred[role] - lumped_actual[role]) / lumped_actual[role] * 100 if lumped_actual[role] > 1e-12 else 0,
            'NMAE_%': nmae_by_group[role],
        })
df_chem = pd.DataFrame(summary_rows)
print(df_chem.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 13. SPEED REPORT — CANTERA VS TUNED ML INFERENCE
# ═══════════════════════════════════════════════════════════════════════════════
# Set these from measured Cantera/PFR timings if available, e.g. from Main_1 or Main_2 logs.
if 'CANTERA_EXIT_SECONDS_PER_RUN' not in globals():
    CANTERA_EXIT_SECONDS_PER_RUN = None  # seconds for one Cantera run to outlet
if 'CANTERA_FULL_PROFILE_SECONDS_PER_RUN' not in globals():
    CANTERA_FULL_PROFILE_SECONDS_PER_RUN = CANTERA_EXIT_SECONDS_PER_RUN  # usually same full PFR solve

import time

speed_reports = {}

# Exit-plane tuned model speed
exit_speed_key = df_results.iloc[0]['Model'].lower().replace(' ', '_')
if exit_speed_key not in models:
    exit_speed_key = list(models.keys())[0]
exit_speed_model = models[exit_speed_key]

_ = exit_speed_model.predict(X_test_s[: min(10, len(X_test_s))])
n_repeats = 5
start = time.perf_counter()
for _ in range(n_repeats):
    _ = exit_speed_model.predict(X_test_s)
elapsed = time.perf_counter() - start

exit_batch_seconds = elapsed / n_repeats
exit_seconds_per_sample = exit_batch_seconds / len(X_test_s)
exit_predictions_per_second = len(X_test_s) / exit_batch_seconds

speed_reports['exit'] = {
    'model': DISPLAY_NAMES.get(exit_speed_key, exit_speed_key),
    'n_test_samples': len(X_test_s),
    'ml_batch_seconds': exit_batch_seconds,
    'ml_seconds_per_exit_prediction': exit_seconds_per_sample,
    'ml_exit_predictions_per_second': exit_predictions_per_second,
    'cantera_seconds_per_run': CANTERA_EXIT_SECONDS_PER_RUN,
    'speedup_vs_cantera': None,
}

print(
    f"Speed exit | {speed_reports['exit']['model']} | n={len(X_test_s)} | "
    f"batch {exit_batch_seconds:.3f}s | {exit_seconds_per_sample * 1e3:.2f} ms/exit | "
    f"{exit_predictions_per_second:.0f} exits/s"
)
if CANTERA_EXIT_SECONDS_PER_RUN is not None and CANTERA_EXIT_SECONDS_PER_RUN > 0:
    speedup = CANTERA_EXIT_SECONDS_PER_RUN / exit_seconds_per_sample
    speed_reports['exit']['speedup_vs_cantera'] = speedup
    print(f"  vs Cantera {CANTERA_EXIT_SECONDS_PER_RUN:.3f}s/run → ~{speedup:.0f}x")
else:
    print("  (Set CANTERA_EXIT_SECONDS_PER_RUN for speedup)")

# Full-profile tuned model speed, if trained
if 'full_profile_model' in globals():
    _ = full_profile_model.predict(X_full_test_s[: min(10, len(X_full_test_s))])
    start = time.perf_counter()
    for _ in range(n_repeats):
        _ = full_profile_model.predict(X_full_test_s)
    elapsed = time.perf_counter() - start

    full_batch_seconds = elapsed / n_repeats
    full_seconds_per_row = full_batch_seconds / len(X_full_test_s)
    full_rows_per_second = len(X_full_test_s) / full_batch_seconds
    n_test_runs = max(full_profile_metrics.get('Test_Runs', 1), 1)
    avg_rows_per_test_run = len(X_full_test_s) / n_test_runs
    ml_seconds_per_profile = full_seconds_per_row * avg_rows_per_test_run

    speed_reports['full_profile'] = {
        'model': DISPLAY_NAMES.get(MODEL_TO_TUNE, MODEL_TO_TUNE),
        'n_test_rows': len(X_full_test_s),
        'n_test_runs': n_test_runs,
        'avg_rows_per_test_run': avg_rows_per_test_run,
        'ml_batch_seconds': full_batch_seconds,
        'ml_seconds_per_axial_row': full_seconds_per_row,
        'ml_rows_per_second': full_rows_per_second,
        'ml_seconds_per_profile_estimate': ml_seconds_per_profile,
        'cantera_seconds_per_profile_run': CANTERA_FULL_PROFILE_SECONDS_PER_RUN,
        'speedup_vs_cantera': None,
    }

    print(
        f"Speed full-profile | {speed_reports['full_profile']['model']} | "
        f"rows={len(X_full_test_s)} | batch {full_batch_seconds:.3f}s | "
        f"{full_seconds_per_row * 1e3:.3f} ms/row | ~{ml_seconds_per_profile * 1e3:.2f} ms/profile"
    )
    if CANTERA_FULL_PROFILE_SECONDS_PER_RUN is not None and CANTERA_FULL_PROFILE_SECONDS_PER_RUN > 0:
        speedup = CANTERA_FULL_PROFILE_SECONDS_PER_RUN / ml_seconds_per_profile
        speed_reports['full_profile']['speedup_vs_cantera'] = speedup
        print(f"  vs Cantera {CANTERA_FULL_PROFILE_SECONDS_PER_RUN:.3f}s/run → ~{speedup:.0f}x")
    else:
        print("  (Set CANTERA_FULL_PROFILE_SECONDS_PER_RUN for speedup)")
else:
    print("Full-profile model not trained — skipped speed (full).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 14. EXPORT MODELS
# ═══════════════════════════════════════════════════════════════════════════════
# Save trained models, scaler, and metadata for inference.

if IF_MODEL_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    run_at = datetime.now().isoformat(timespec="seconds")
    artifact_path = EXPORT_DIR / "tree_model_tuned_exit_full.joblib"
    
    artifact = {
        'models': models,
        'scaler_X': scaler,
        'feature_cols': feature_cols,
        'target_cols': target_cols,
        'label_encoder': label_encoder,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'X_train_s': X_train_s,
        'X_test_s': X_test_s,
        'config': config,
        'data_file': DATA_FILE,
        'run_at': run_at,
    }
    if 'speed_reports' in globals():
        artifact['speed_reports'] = speed_reports

    if 'full_profile_model' in globals():
        artifact['full_profile'] = {
            'model': full_profile_model,
            'models': full_profile_models,
            'scaler_X': full_scaler,
            'label_encoder': full_label_encoder,
            'feature_cols': feature_cols,
            'target_cols': target_cols,
            'metrics': full_profile_metrics,
            'X_train': X_full_train,
            'X_test': X_full_test,
            'y_train': y_full_train,
            'y_test': y_full_test,
            'X_train_s': X_full_train_s,
            'X_test_s': X_full_test_s,
        }
    
    joblib.dump(artifact, artifact_path)
    print(f"Exported {artifact_path.name} | exit keys={list(models.keys())} | "
          f"full_profile={'full_profile' in artifact} | feat={len(feature_cols)} tgt={len(target_cols)}")
else:
    print("Model export disabled.")

---

## Summary

This notebook tunes one selected tree model for two related surrogate tasks:

1. Inlet→outlet / exit-plane prediction: one sample per simulation run at maximum `relative_position`.
2. Full axial / PFR evolution prediction: all axial rows are used, with `relative_position` as an input feature.
3. Run-level full-profile split: full-profile train/test data are split by simulation run to avoid leakage between axial points from the same PFR profile.
4. Hyperparameter tuning: `BayesSearchCV` (exit-plane only) tunes `MODEL_TO_TUNE`; optional early stopping via `TUNING_PATIENCE` / `TUNING_MIN_DELTA`.
5. Full-profile training: the axial/PFR model reuses the same hyperparameters as the exit-plane tuned model (no second search).
6. Speed report: ML inference timing for exit prediction and, when enabled, full-profile prediction; optional Cantera/PFR speedups are printed when measured baseline times are set.
7. Export: the `.joblib` artifact includes the tuned exit model and, when enabled, the full-profile model plus its scaler, metrics, and speed report.

Use `Main_4_train_and_evaluate_tree_models_IO.ipynb` for fast default-parameter baseline comparison across model families.